# 02 · Distributions & Fat Tails
**Goal:** see *why real returns are not a tidy bell curve* — they have **fat tails** (extreme moves happen far more often than 'normal' predicts). This one fact drives the whole risk project.

> Maps to: projects **07** (risk) and **30** (bars). KB §1.4.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(1)

## 1. The normal distribution (bell curve)
The **normal** is the default assumption in finance because it's simple: just a *mean* (center) and *standard deviation* (width). Let's look at one.

In [ ]:
x = np.linspace(-5, 5, 400)
plt.figure(figsize=(8,3))
plt.plot(x, stats.norm.pdf(x), label='Normal')
plt.plot(x, stats.t.pdf(x, df=3), label='Student-t (df=3, fat-tailed)')
plt.title('Normal vs a fat-tailed distribution'); plt.legend(); plt.show()

Notice the orange **Student-t** curve sits *higher in the tails* (far left/right). Those rare-but-not-that-rare extreme outcomes are exactly what blow up naive risk models.

## 2. Real returns have fat tails
Let's compare *normal* returns with *fat-tailed* returns that have the same volatility, and measure the tail-fatness with **excess kurtosis** (0 = normal; higher = fatter) and **skewness** (asymmetry).

In [ ]:
n = 5000
normal_rets = rng.normal(0, 0.01, n)
fat_rets = stats.t.rvs(df=3, size=n, random_state=rng) * 0.01 / np.sqrt(3/(3-2))

for name, r in [('normal', normal_rets), ('fat-tailed', fat_rets)]:
    print(f'{name:11s}  std={r.std():.4f}  skew={stats.skew(r):+.2f}  excess_kurtosis={stats.kurtosis(r):6.2f}')

In [ ]:
plt.figure(figsize=(8,3))
bins = np.linspace(-0.06, 0.06, 60)
plt.hist(normal_rets, bins=bins, alpha=.5, label='normal', density=True)
plt.hist(fat_rets, bins=bins, alpha=.5, label='fat-tailed', density=True)
plt.title('Same volatility, very different tails'); plt.legend(); plt.yscale('log')
plt.show()

On a **log scale** the fat-tailed sample clearly has more mass way out in the tails — those are the crash/spike days.

## 3. The Jarque–Bera normality test
Instead of eyeballing, we test it. **Jarque–Bera** combines skew + kurtosis into a p-value:

- p **> 0.05** → can't reject normal (looks roughly normal)
- p **< 0.05** → reject normal (significantly non-normal)

In [ ]:
for name, r in [('normal', normal_rets), ('fat-tailed', fat_rets)]:
    jb, p = stats.jarque_bera(r)
    verdict = 'looks normal' if p > 0.05 else 'NOT normal'
    print(f'{name:11s}  JB stat={jb:9.1f}  p-value={p:.2e}  -> {verdict}')

## 4. More examples: how fat is 'fat'?
The Student-t's `df` (degrees of freedom) controls tail fatness: small df = very heavy tails; large df ≈ normal. Watch the kurtosis and 'worst day' shrink as df grows.

In [ ]:
print(f'{"df":>4}{"excess_kurt":>13}{"worst of 5000":>15}')
for df in [3, 5, 10, 30, 100]:
    sample = stats.t.rvs(df=df, size=5000, random_state=np.random.default_rng(df)) * 0.01
    print(f'{df:4d}{stats.kurtosis(sample):13.2f}{sample.min()*100:13.2f}%')

Notice how a `df=3` world has both fatter measured kurtosis **and** a much worse single worst day than a `df=100` (near-normal) world — even at identical volatility. That extra tail is exactly the risk a Normal model misses.

### 🧪 Try it yourself
1. Set `df=3` to `df=2.5` — kurtosis becomes huge/unstable (the 4th moment barely exists). Real markets are often around df 3–6.
2. Run Jarque–Bera on a *small* sample (`n=50`) of truly fat-tailed data — it may *fail* to reject normality. Tail tests need lots of data.
3. Compute the 1% quantile of `normal_rets` vs `fat_rets` — the fat one's is much worse. That quantile *is* Value-at-Risk (see the VaR notebook in `07-var-es-risk-engine/concepts/`).

**You should see:** the fat-tailed sample has large excess kurtosis and a tiny Jarque–Bera p-value (reject normal), even though its standard deviation matches the normal one. Volatility alone does **not** capture tail risk.

### In the projects
- Student-t VaR (fat tails) → project **07** `src/var.py:parametric_var(dist='t')`.
- Jarque–Bera on bar returns → project **30** `src/bars.py:bar_return_stats`.